In [1]:
!git clone https://github.com/tuananhpham-vnu/ADAPT.git

Cloning into 'ADAPT'...
remote: Enumerating objects: 1006, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 1006 (delta 82), reused 149 (delta 36), pack-reused 794 (from 1)
Receiving objects: 100% (1006/1006), 131.98 MiB | 33.86 MiB/s, done.
Resolving deltas: 100% (406/406), done.


In [2]:
%cd ADAPT/
!ls

/kaggle/working/ADAPT
adapt.ipynb	  embedder	   ReAct			 scripts
adapt_tracing.py  environment.yml  README.md			 src
agentdriver	  git		   requirements-agentdriver.txt  survey
algo		  _guidance	   requirements-aqua.txt	 tests
configs		  LICENSE	   requirements.txt
EhrAgent	  make.ps1	   sanity_check.py


In [3]:
%%capture
!pip install -r requirements.txt

In [4]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [5]:
!bash scripts/run_specificity_ap.sh all


>> stage [preflight] -> outputs/specificity/kaggle/logs/preflight.log
===== preflight =====
  torch          2.10.0+cu128
  transformers   5.0.0
  numpy          2.0.2
  sklearn        1.6.1
  cuda available True | devices 2
  data ok        ReAct/database/strategyqa_train_filtered.json
  data ok        ReAct/database/strategyqa_dev.json
  data ok        ReAct/database/strategyqa_train_paragraphs.json
  data ok        ReAct/database/strategyqa_train.json
  data ok        ReAct/database/strategyqa_test.json
  train pool     2821 rows, need 144, labelled=True
  test pool      229 rows, need 229 (bank test + fresh language test), labelled=True
  valid pool     490 rows, need 489, labelled=False
  per_query      5 poison key(s) per trigger, TOP_K=5
  poisoning      120/9251 paragraphs = 1.30% of the corpus
===== preflight ok =====

>> stage [models] -> outputs/specificity/kaggle/logs/models.log
===== models (needs internet) =====
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 273

In [6]:
# !python -m src.agentpoison.strategyqa index \
#   --provider deepseek \
#   --device cuda \
#   --num-queries 10 \
#   --batch-size 32 \
#   --index ReAct/database/embeddings/agentpoison_dpr


In [7]:
# !python -m src.agentpoison.phases optimize \
#   --agent qa \
#   --retriever-model dpr-ctx_encoder-single-nq-base \
#   --num-iter 5 \
#   --num-cand 20 \
#   --num-grad-iter 3 \
#   --opt-batch-size 16 \
#   --trigger-output results/triggers/qa-dpr-smoke.json


In [8]:
# !python -m json.tool results/triggers/qa-dpr-smoke.json

In [9]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [10]:
# !python -m src.agentpoison.phases optimize \
#   --agent qa \
#   --retriever-model dpr-ctx_encoder-single-nq-base \
#   --num-iter 1000 \
#   --num-cand 100 \
#   --num-grad-iter 30 \
#   --opt-batch-size 32 \
#   --trigger-output results/triggers/qa-dpr-ap.json

In [11]:
# RUN_DIR=kaggle/working/ADAPT/results/ssh_smoke_01
# TRIGGER=kaggle/working/ADAPT/results/triggers/qa-dpr-ap.json

In [12]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [13]:
# !python -m src.agentpoison.phases prepare \
#   --run-dir "$RUN_DIR" \
#   --trigger-file "$TRIGGER" \
#   --provider deepseek \
#   --device cuda \
#   --num-queries 1 \
#   --batch-size 16 \
#   --index ReAct/database/embeddings/agentpoison_dpr \
#   --repeats 1


In [14]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [15]:
# !python -m src.agentpoison.phases retrieve --run-dir "$RUN_DIR"

In [16]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [17]:
# !python -m src.agentpoison.phases infer \
#   --run-dir "$RUN_DIR" \
#   --provider deepseek

In [18]:
    # !python -m src.agentpoison.phases infer \
    #   --run-dir "$RUN_DIR" \
    #   --provider deepseek \
    #   --retry-errors

In [19]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [20]:
# !python -m src.agentpoison.phases evaluate --run-dir "$RUN_DIR"